# Lab 5: Non-negative Matrix Factorization

In this lab we implement NMF from scratch with the multiplicative updates of the lecture, for the Frobenius and the Kullback–Leibler losses, and apply it to faces. The effect of the penalties and topics in text are homework.

In [ ]:
%matplotlib inline
import json
import base64
import numpy as np
import matplotlib.pyplot as plt
plt.style.use("seaborn-v0_8")

## Helpers

The three functions below are given. Run the three cells and move on. The faces are the lecture's file `faces.json`, and the posts of the homework are in `newsgroups.json`.

**`load_faces`** loads 400 photos of faces (Olivetti dataset), 32 x 32 pixels each, as a matrix with **one face per column**, as in the lecture.

In [ ]:
def load_faces(path="faces.json"):
    """Return X, a (1024, 400) array: column j is face j, 32 x 32 = 1024 pixels with values between 0 and 1."""
    with open(path) as f:
        data = json.load(f)
    # the pixels are stored as bytes (0 to 255), one face after the other
    pixels = np.frombuffer(base64.b64decode(data["pixels"]), dtype=np.uint8)
    faces = pixels.reshape(data["n"], data["side"] * data["side"]) / 255
    return faces.T

**`load_topics`** loads 1000 posts from 4 discussion groups (20 Newsgroups dataset) as a word-by-post matrix: entry $(i, j)$ is the weight of word $i$ in post $j$ (tf-idf), with **one post per column** (homework).

In [ ]:
def load_topics(path="newsgroups.json"):
    """Return X, a (600, 1000) array of word weights (one post per column), and vocab, the list of the 600 words."""
    with open(path) as f:
        data = json.load(f)
    vocab = data["vocab"]
    X = np.zeros((len(vocab), len(data["rows"])))
    # each post is stored as (indices of its words, weights of these words)
    for j, (words, weights) in enumerate(data["rows"]):
        X[words, j] = weights
    return X, vocab

**`plot_images`** draws the columns of a matrix as 32 x 32 images: faces, or the parts found by NMF.

In [ ]:
def plot_images(images, title, n_cols=8):
    """Draw every column of images, a (1024, k) array, as a 32 x 32 image, n_cols images per row."""
    n_images = images.shape[1]
    n_rows = int(np.ceil(n_images / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(1.6 * n_cols, 1.7 * n_rows), squeeze=False)
    for k, ax in enumerate(axes.ravel()):
        ax.axis("off")
        if k < n_images:
            ax.imshow(images[:, k].reshape(32, 32), cmap="gray")
    fig.suptitle(title)
    plt.show()

## Data

Each face is a column of $X$: a non-negative vector of 1024 pixel intensities.

In [ ]:
X = load_faces()
print("X:", X.shape, "| smallest value:", X.min().round(3), "| largest value:", X.max().round(3))
plot_images(X[:, :16], "16 faces of the dataset")

## 1. NMF with the multiplicative updates

**The three matrices.**
- $X$ ($p \times n$): the data, **one face image per column**: $p = 1024$ pixels (32 x 32) and $n = 400$ faces.
- $W$ ($p \times r$): the **parts**, also one image per column: $r$ images of $p$ pixels, shared by all faces.
- $H$ ($r \times n$): the **weights**, one column per face: column $j$ says how much of each of the $r$ parts face $j$ uses. These $r$ numbers are the new, short description of face $j$.

NMF looks for non-negative $W$ and $H$ such that
$$
X \approx W H .
$$
$WH$ ($p \times n$) is the **reconstruction** of the faces: column $j$ of $WH$ rebuilds face $j$ as a weighted sum of the $r$ parts, with the weights of column $j$ of $H$.

**The number of parts $r$** is chosen by us before the fit, much smaller than $p$ and $n$ (here $r = 16$). The fit never changes $r$: it finds the best $W$ and $H$ for this $r$. A smaller $r$ gives a shorter description of each face ($r$ numbers instead of $p$ pixels) but a worse reconstruction (see "Choosing $r$" below).

**The goal.** Describe every face as a sum of a few parts. Since nothing is negative, the parts can only be added, never subtracted, so they can be read: a part brightens some regions of a face, and a weight says how much of this part a face uses. The same model finds topics in text (a part is a topic, see the homework), notes in music, or materials in satellite images.

**The loss.** We want $WH$ close to $X$: this is the Frobenius loss, plus three optional penalties from the lecture:
$$
L(W, H) = \underbrace{\frac{1}{2}\sum_{i,j} \big(X - WH\big)_{ij}^2}_{\text{Frobenius loss}}
+ \underbrace{\frac{\mu}{2}\sum_{k,j} H_{kj}^2}_{\text{keeps } H \text{ small}}
+ \underbrace{\lambda\sum_{k,j} H_{kj}}_{\text{makes } H \text{ sparse}}
+ \underbrace{\frac{\nu}{2}\sum_{i,k} W_{ik}^2}_{\text{keeps } W \text{ small}} .
$$
- **Frobenius loss**: half the sum of the squared differences between the faces and their reconstruction, over all pixels and all faces;
- $\mu$: penalizes the squared weights, so the weights stay small;
- $\lambda$: penalizes the sum of the weights (they are all $\ge 0$), which pushes many of them to exactly 0: each face then uses only a few parts;
- $\nu$: penalizes the squared values of the parts, so the parts stay small.

With $\mu = \nu = \lambda = 0$ there is no penalty, only the Frobenius loss.

**The updates.** Starting from random positive $W$ and $H$, we repeat
$$
W \leftarrow W \odot \frac{X H^T}{W(HH^T + \nu I_r)}, \qquad
H \leftarrow H \odot \frac{\left[W^T X - \lambda \mathbf{1}\right]_+}{(W^T W + \mu I_r) H},
$$
where:
- $\odot$ multiplies entry by entry, and the fraction divides entry by entry;
- $I_r$ is the $r \times r$ identity matrix (1 on the diagonal, 0 elsewhere): adding $\nu I_r$ adds $\nu$ to the diagonal of $HH^T$, and in NumPy it is `nu * np.eye(r)`;
- $\lambda \mathbf{1}$ is the $r \times n$ matrix full of $\lambda$: it subtracts $\lambda$ from every entry of $W^T X$, and in NumPy it is simply `W.T @ X - lam`;
- $[\,\cdot\,]_+ = \max(\cdot, 0)$: without it, a large $\lambda$ would make $H$ negative;
- in the code, $10^{-10}$ is added to every denominator so that we never divide by 0 (for example when all the weights of a face become 0).

With $\mu = \nu = \lambda = 0$, these are the Lee and Seung rules. Each update never increases the loss, but convergence is slow.

**Exercise 1.** Complete `NMF`: `loss`, `update` (one update of $W$, then of $H$) and `fit` (the random start and the loop of updates).

In [ ]:
class NMF:
    """Non-negative matrix factorization X ~ W H with the Frobenius loss and the penalties of the lecture,
    fitted with the regularized multiplicative updates."""

    def __init__(self, r, mu=0., nu=0., lam=0., n_iter=300, seed=0):
        self.r = r                # number of parts (columns of W)
        self.mu = mu              # penalty mu/2 ||H||^2: keeps H small
        self.nu = nu              # penalty nu/2 ||W||^2: keeps W small
        self.lam = lam            # penalty lam ||H||_1: makes H sparse
        self.n_iter = n_iter      # number of updates of W and H
        self.seed = seed          # seed of the random start
        self.W_ = None            # (p, r) array: the parts
        self.H_ = None            # (r, n) array: the weights
        self.loss_history_ = []   # loss after every update

    def loss(self, X):
        """Penalized Frobenius loss of the current W_ and H_."""
        # TODO
        # Step 1: Frobenius loss 1/2 ||X - W H||^2, with W, H = self.W_, self.H_:
        #         half the sum of the squared entries of X - W @ H (np.sum)
        # Step 2: penalties mu/2 ||H||^2 + lam ||H||_1 + nu/2 ||W||^2 (||H||_1 is np.sum(H), since H >= 0)
        # Step 3: return the Frobenius loss + the penalties

    def update(self, X, W, H):
        """One multiplicative update of W, then of H. Returns the new W and H."""
        # TODO
        # Step 1: identity matrix I = np.eye(self.r)
        # Step 2: update W: W * (X H^T) / (W (H H^T + nu I) + 1e-10)   (1e-10 avoids dividing by 0)
        # Step 3: update H with the new W: H * max(W^T X - lam, 0) / ((W^T W + mu I) H + 1e-10)   (np.maximum)
        # Step 4: return W, H

    def fit(self, X):
        """Fit W_ and H_ to the (p, n) non-negative data X."""
        # TODO
        # Step 1: random positive start: rng = np.random.RandomState(self.seed),
        #         W = rng.rand(p, self.r) and H = rng.rand(self.r, n), with p, n = X.shape
        # Step 2: an empty self.loss_history_
        # Step 3: repeat self.n_iter times:
        #     a) W, H = self.update(X, W, H)
        #     b) store W and H in self.W_ and self.H_, and append self.loss(X) to self.loss_history_
        return self

Fit NMF with $r = 16$ parts and no penalty. The relative error $\|X - WH\| / \|X\|$ measures how well $WH$ rebuilds the faces:

In [ ]:
model = NMF(r=16, n_iter=500)
model.fit(X)

history = np.array(model.loss_history_)
print("Loss went down at every update:", bool(np.all(np.diff(history) <= 1e-9 * history[:-1])))  # up to rounding errors
print(f"Relative error: {np.linalg.norm(X - model.W_ @ model.H_) / np.linalg.norm(X):.3f}")

plt.figure(figsize=(8, 4))
plt.plot(history)
plt.yscale("log")
plt.xlabel("update")
plt.ylabel("loss")
plt.title("Frobenius loss")
plt.show()

The 16 parts found by NMF (columns of $W$), and the reconstruction $WH$ of a few faces:

In [ ]:
plot_images(model.W_, "NMF: the 16 parts (columns of W)")

faces = [0, 50, 100, 150, 200, 250, 300, 350]
plot_images(X[:, faces], "Original faces")
plot_images(model.W_ @ model.H_[:, faces], "Reconstruction W H, Frobenius loss")

The loss goes down at every update, quickly at first and then slowly ("monotone and slow").
NMF parts are **non-negative and additive**: each part brightens some regions of a face (eyes, glasses, nose, mouth), and every face is rebuilt by adding the 16 parts with non-negative weights. The reconstructions keep the main features of each face (shape of the face, beard, expression) but are blurred, and finer details such as the glasses are mostly lost: each face is now described by 16 numbers instead of 1024 pixels.

### Choosing $r$

How many parts? The relative error for several numbers of parts $r$:

In [ ]:
rs = [2, 4, 8, 16, 32, 48]
errors = []
for r in rs:
    model_r = NMF(r=r)
    model_r.fit(X)
    errors.append(np.linalg.norm(X - model_r.W_ @ model_r.H_) / np.linalg.norm(X))

plt.figure(figsize=(8, 4))
plt.plot(rs, errors, marker="o")
plt.xlabel("r")
plt.ylabel("relative error")
plt.show()

The curve never rises: a larger $r$ always fits better, so the error alone never chooses $r$ (lecture: "Choosing r"). Here there is no clear elbow; in practice, $r$ comes from held-out entries, from the stability of the parts across restarts, or from domain knowledge.

## 2. Kullback–Leibler loss

The Kullback–Leibler divergence between $X$ and $Y = WH$ is
$$
D(X \,\|\, WH) = \sum_{i,j} \Big( x_{ij} \log\frac{x_{ij}}{y_{ij}} - x_{ij} + y_{ij} \Big),
$$
and the same recipe as for Frobenius gives its multiplicative updates (lecture: "The other losses, same recipe"):
$$
W \leftarrow W \odot \frac{\left(X \oslash WH\right) H^T}{\mathbf{1} H^T}, \qquad
H \leftarrow H \odot \frac{W^T \left(X \oslash WH\right)}{W^T \mathbf{1}},
$$
where $\oslash$ is the entry-by-entry division and $\mathbf{1}$ the matrix of ones of the size of $X$.
In NumPy, $\mathbf{1} H^T$ has the sums of the rows of $H$ in every row, `H.sum(axis=1)`, and $W^T \mathbf{1}$ has the sums of the columns of $W$ in every column, `W.sum(axis=0)[:, None]`.
(No penalties here.)

**Exercise 2.** Complete `NMFKL`. It inherits `__init__` and `fit` from `NMF`: the random start and the loop are the same, and `fit` calls `self.update` and `self.loss`, so only these two methods change.

In [ ]:
class NMFKL(NMF):
    """Non-negative matrix factorization X ~ W H with the Kullback-Leibler loss (no penalties)."""

    def loss(self, X):
        """Kullback-Leibler divergence D(X || W H) of the current W_ and H_."""
        # TODO
        # Step 1: Y = self.W_ @ self.H_ + 1e-10
        # Step 2: return the sum of X * log((X + 1e-10) / Y) - X + Y   (1e-10 makes x log x = 0 when x = 0)

    def update(self, X, W, H):
        """One multiplicative update of W, then of H, for the KL loss. Returns the new W and H."""
        # TODO
        # Step 1: update W: W * ((X / (W H + 1e-10)) H^T) / (H.sum(axis=1) + 1e-10)
        # Step 2: update H with the new W: H * (W^T (X / (W H + 1e-10))) / (W.sum(axis=0)[:, None] + 1e-10)
        # Step 3: return W, H

Fit NMF with the KL loss, and compare the reconstructions $WH$ of a few faces with both losses:

In [ ]:
model_kl = NMFKL(r=16, n_iter=500)
model_kl.fit(X)

history_kl = np.array(model_kl.loss_history_)
print("Loss went down at every update:", bool(np.all(np.diff(history_kl) <= 1e-9 * history_kl[:-1])))
print(f"Relative error, Frobenius: {np.linalg.norm(X - model.W_ @ model.H_) / np.linalg.norm(X):.3f}")
print(f"Relative error, KL:        {np.linalg.norm(X - model_kl.W_ @ model_kl.H_) / np.linalg.norm(X):.3f}")

plot_images(X[:, faces], "Original faces")
plot_images(model.W_ @ model.H_[:, faces], "Reconstruction W H, Frobenius loss")
plot_images(model_kl.W_ @ model_kl.H_[:, faces], "Reconstruction W H, KL loss")

The KL loss also goes down at every update. Both losses rebuild the faces from 16 parts with a similar error: they measure the gap between $X$ and $WH$ differently (squares for Frobenius, ratios for KL), and the lecture links each loss to a noise model.

## Homework

### Effect of the penalties

The penalty $\lambda\|H\|_1$ should put zeros in $H$, so that each face uses only a few parts. But a penalty on $H$ alone can be escaped: $H$ gets smaller while $W$ gets larger, with the same product $WH$. So we keep $W$ small with $\nu = 10$, and increase $\lambda$:

In [ ]:
for lam in [0, 1, 3, 10]:
    sparse_model = NMF(r=16, nu=10, lam=lam)
    sparse_model.fit(X)
    zeros = np.mean(sparse_model.H_ == 0)
    error = np.linalg.norm(X - sparse_model.W_ @ sparse_model.H_) / np.linalg.norm(X)
    print(f"lambda = {lam:2d}: zeros in H = {zeros:.0%}, relative error = {error:.3f}")

As $\lambda$ grows, more and more weights in $H$ are exactly 0, so each face uses fewer parts, and the fit gets worse: with $\lambda = 10$, 94% of the weights are 0 and the relative error goes from 0.121 to 0.207 (lecture: "λ up: more zeros in H, a worse fit").

### Topics in text

$X$ is now a word-by-post matrix: 600 words, 1000 posts from 4 discussion groups (computer graphics, cars, hockey, space). With $r = 4$, each column of $W$ is a **topic**: a non-negative weight for every word.

**Exercise 3 (homework).** Complete `top_words`.

In [ ]:
def top_words(W, vocab, n_words=8):
    """For every column of W (a topic), the n_words words with the largest weights.

    Parameters:
    -----------
    W: (num_words, r) np.array
    vocab: list of the num_words words
    n_words: int

    Returns:
    --------
    topics: list of r lists of n_words words
    """
    # TODO
    # for every column k of W:
    #     a) indices of the words sorted from the largest to the smallest weight: np.argsort(W[:, k])[::-1]
    #     b) keep the first n_words indices, and turn them into words with vocab
    # return the list of the r lists of words

In [ ]:
X_text, vocab = load_topics()
print("X_text:", X_text.shape)

text_model = NMF(r=4)
text_model.fit(X_text)
for k, words in enumerate(top_words(text_model.W_, vocab)):
    print(f"Topic {k}:", ", ".join(words))

The 4 topics match the 4 groups: computer graphics (graphics, image, file), hockey (team, game, hockey), space (space, nasa, launch) and cars (car, dealer, engine). NMF was never told which group a post comes from: it found the topics from the words alone. Common words (know, think, like, just) appear too, since they are used in every group.